In [1]:
"""
Stimulation-Accuracy Behavioral Analysis
Sessions: S2 (Session 2 - 4-digit span) and S3 (Session 3 - 3-digit span)
DBS device: Medtronic Percept PC | Target: ATN bilateral | LFP channel: Left
Analyses:
  1. Overall accuracy: Stim ON vs Stim OFF (all trials, both sessions)
  2. Accuracy by difficulty: 2 / 3 / 4 digits (Stim ON vs OFF where applicable)
  3. S2 hard trials (4-digit only): Stim ON vs OFF accuracy & RT
  4. Stimulation timing window: Stim concentrated in ENCODING vs RECALL phase
  5. mA dose vs accuracy (continuous stimulation level)
  6. Session-level accuracy overview
"""

# %matplotlib inline
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from datetime import datetime
from scipy import stats

# ──────────────────────────────────────────────
# FILE PATHS  (update if needed)
# ──────────────────────────────────────────────
JSON_S2 = r'C:\Users\ASSUS\ATN\Digit Span Backwards\Data\Neural Data\DBS ATN DSB Case 1\D. Siragusa\3.5.26\Time stamp 1433\Report_Json_Session_Report_20260305T151703.json'
JSON_S3 = r'C:\Users\ASSUS\ATN\Digit Span Backwards\Data\Neural Data\DBS ATN DSB Case 1\D. Siragusa\3.5.26\Time stamp 1441\Report_Json_Session_Report_20260305T151912.json'
BEH_S2  = r'C:\Users\ASSUS\ATN\Digit Span Backwards\Data\Eprime Data\Digit Span Backwards v3.2\DigitSpanBackward v3.3-6-2-Scores.csv'
BEH_S3  = r'C:\Users\ASSUS\ATN\Digit Span Backwards\Data\Eprime Data\Digit Span Backwards v3.2\DigitSpanBackward v3.3-6-3-Scores.csv'
EVENTS  = r'C:\Users\ASSUS\ATN\Digit Span Backwards\Data\Eprime Data\Digit Span Backwards v3.2\Events.csv'   # Events from Session 3

# ──────────────────────────────────────────────
# SYNC CONSTANTS
# Derived from: first stim spike in TimeDomain = task start
# neural_sample = (RTTime_ms - SYNC_OFFSET) / 4.0
# neural_abs = t0_neural + sample / SR
# ──────────────────────────────────────────────
SYNC_OFFSET_S2 = 45122.0   # ms  (RTTime=73730 -> sample 7152 -> abs 1772739640.608)
SYNC_OFFSET_S3 = -225.0    # ms  (RTTime=54415 -> sample 13660 -> abs 1772740103.640)
T0_S2          = 1772739612.0   # neural start abs (seconds)
T0_S3          = 1772740049.0
SR             = 250.0          # Hz
STIM_THRESH_MA = 2.0            # mA threshold for Stim ON


# ──────────────────────────────────────────────
# HELPERS
# ──────────────────────────────────────────────
def rttime_to_abs(rttime_ms, t0, sync_offset):
    sample = (rttime_ms - sync_offset) / 4.0
    return t0 + sample / SR

def build_lfp_df(j):
    records = []
    for entry in j['BrainSenseLfp']:
        t0 = datetime.fromisoformat(
            entry['FirstPacketDateTime'].replace('Z', '+00:00')).timestamp()
        sr_lfp = entry['SampleRateInHz']
        for k, d in enumerate(entry['LfpData']):
            records.append({
                't_abs': t0 + k / sr_lfp,
                'mA_L':  d.get('Left',  {}).get('mA', 0),
                'mA_R':  d.get('Right', {}).get('mA', 0)
            })
    return pd.DataFrame(records).sort_values('t_abs').reset_index(drop=True)

def get_mean_mA(lfp_df, t_start, t_end):
    mask = (lfp_df['t_abs'] >= t_start) & (lfp_df['t_abs'] <= t_end)
    sub = lfp_df[mask]
    if len(sub) == 0:
        idx = (lfp_df['t_abs'] - (t_start + t_end) / 2).abs().idxmin()
        return float(lfp_df.loc[idx, 'mA_L'])
    return float(sub['mA_L'].mean())

def cohens_d(a, b):
    if len(a) < 2 or len(b) < 2:
        return np.nan
    pooled = np.sqrt(((len(a)-1)*np.std(a,ddof=1)**2 +
                      (len(b)-1)*np.std(b,ddof=1)**2) / (len(a)+len(b)-2))
    return (np.mean(a) - np.mean(b)) / pooled if pooled > 0 else np.nan

def safe_sem(x):
    return float(np.std(x, ddof=1) / np.sqrt(len(x))) if len(x) > 1 else 0.0


# ──────────────────────────────────────────────
# LOAD DATA
# ──────────────────────────────────────────────
with open(JSON_S2) as f: j2 = json.load(f)
with open(JSON_S3) as f: j3 = json.load(f)

beh2 = pd.read_csv(BEH_S2)
beh3 = pd.read_csv(BEH_S3)
ev   = pd.read_csv(EVENTS, encoding='utf-8-sig')   # Session 3 events

lfp2 = build_lfp_df(j2)
lfp3 = build_lfp_df(j3)


# ──────────────────────────────────────────────
# BUILD TRIAL TABLE
# ──────────────────────────────────────────────
def build_trials(beh_df, lfp_df, t0, sync_offset, session_label, ev_df=None):
    """
    Returns one row per trial with timing, stim, and phase-level stim info.
    ev_df: Events.csv (only available for S3; used for phase timing).
    """
    cols = ['Block', 'Trial', 'CurrentSpanSize[Trial]',
            'CollectResponse.ACC', 'CollectResponse.RT',
            'CollectResponse.RTTime', 'CollectResponse.OnsetTime',
            'Fixation.OnsetTime', 'Feedback.OnsetTime']
    t = (beh_df[cols]
         .drop_duplicates(subset=['Block', 'Trial'])
         .dropna(subset=['CollectResponse.ACC'])
         .copy()
         .reset_index(drop=True))
    t.rename(columns={
        'CurrentSpanSize[Trial]':  'num_digits',
        'CollectResponse.ACC':     'acc',
        'CollectResponse.RT':      'rt_ms',
        'CollectResponse.RTTime':  'rttime',
        'CollectResponse.OnsetTime':'choice_onset_ms',
        'Fixation.OnsetTime':      'fix_onset_ms',
        'Feedback.OnsetTime':      'fb_onset_ms'
    }, inplace=True)

    t['session']         = session_label
    t['trial_idx']       = np.arange(1, len(t)+1)

    # Absolute timestamps
    t['fix_abs']         = t['fix_onset_ms'].apply(lambda x: rttime_to_abs(x, t0, sync_offset))
    t['choice_start_abs']= t['choice_onset_ms'].apply(lambda x: rttime_to_abs(x, t0, sync_offset))
    t['choice_end_abs']  = t['rttime'].apply(lambda x: rttime_to_abs(x, t0, sync_offset))

    results = []
    for _, row in t.iterrows():
        enc_start = row['fix_abs']
        enc_end   = row['choice_start_abs']
        rec_start = row['choice_start_abs']
        rec_end   = row['choice_end_abs']
        trial_start = enc_start
        trial_end   = rec_end

        # Overall stim during trial
        mA_trial   = get_mean_mA(lfp_df, trial_start, trial_end)
        # Phase-level
        mA_enc     = get_mean_mA(lfp_df, enc_start, enc_end)
        mA_rec     = get_mean_mA(lfp_df, rec_start, rec_end)

        stim_on        = mA_trial >= STIM_THRESH_MA
        stim_enc_on    = mA_enc   >= STIM_THRESH_MA
        stim_rec_on    = mA_rec   >= STIM_THRESH_MA

        enc_dur = max(enc_end - enc_start, 1e-6)
        rec_dur = max(rec_end - rec_start, 1e-6)

        results.append({
            'session':       session_label,
            'block':         int(row['Block']),
            'trial':         int(row['Trial']),
            'trial_idx':     int(row['trial_idx']),
            'num_digits':    int(row['num_digits']),
            'acc':           int(row['acc']),
            'rt_ms':         float(row['rt_ms']),
            'mean_mA':       mA_trial,
            'mA_enc':        mA_enc,
            'mA_rec':        mA_rec,
            'stim_on':       stim_on,
            'stim_enc_on':   stim_enc_on,
            'stim_rec_on':   stim_rec_on,
            'trial_dur_s':   trial_end - trial_start,
            'enc_dur_s':     enc_dur,
            'rec_dur_s':     rec_dur,
        })
    return pd.DataFrame(results)

trials_s2 = build_trials(beh2, lfp2, T0_S2, SYNC_OFFSET_S2, 'S2')
trials_s3 = build_trials(beh3, lfp3, T0_S3, SYNC_OFFSET_S3, 'S3')
all_trials = pd.concat([trials_s2, trials_s3], ignore_index=True)

print("="*60)
print("TRIAL SUMMARY")
print("="*60)
for sess, df in [('S2', trials_s2), ('S3', trials_s3)]:
    print(f"\n{sess}: {len(df)} trials | Digits: {sorted(df['num_digits'].unique())}")
    print(f"  Stim ON: {df['stim_on'].sum()} | OFF: {(~df['stim_on']).sum()}")
    print(f"  Overall accuracy: {df['acc'].mean()*100:.1f}%")
    by_stim = df.groupby('stim_on')['acc'].mean()
    for s, v in by_stim.items():
        print(f"    Stim {'ON' if s else 'OFF'}: {v*100:.1f}% ({(df['stim_on']==s).sum()} trials)")


# ──────────────────────────────────────────────
# PLOT STYLE HELPERS
# ──────────────────────────────────────────────
COL_ON  = '#D6604D'   # red  - Stim ON
COL_OFF = '#2166AC'   # blue - Stim OFF
COL_S2  = '#1B7837'   # green - Session 2
COL_S3  = '#762A83'   # purple - Session 3

def bar_with_sem(ax, positions, means, sems, colors, labels=None, width=0.5):
    bars = ax.bar(positions, means, width=width, color=colors,
                  edgecolor='black', linewidth=1.0, zorder=3)
    ax.errorbar(positions, means, yerr=sems, fmt='none',
                color='black', capsize=5, capthick=1.5, linewidth=1.5, zorder=4)
    if labels:
        ax.set_xticks(positions)
        ax.set_xticklabels(labels, fontsize=10)
    return bars

def add_sig_bracket(ax, x1, x2, y, p, d, color='red'):
    """Draw significance bracket with p-value and Cohen's d."""
    h = y * 0.04
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], color='black', linewidth=1.2)
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    d_str = f"d={d:.2f}" if not np.isnan(d) else ""
    ax.text((x1+x2)/2, y+h*1.1,
            f"{sig}\np={p:.3f}\n{d_str}",
            ha='center', va='bottom', fontsize=8,
            color=color if sig != 'ns' else 'black',
            fontweight='bold' if sig != 'ns' else 'normal')


# ══════════════════════════════════════════════
# FIGURE 1: Overall Stim ON vs OFF Accuracy
# ══════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(15, 6), facecolor='white')
fig.suptitle('Stimulation Effect on Accuracy — All Sessions',
             fontsize=14, fontweight='bold', y=1.01)

# Panel A: Stim ON vs OFF accuracy (all trials, pooled)
ax = axes[0]
for stim_val, color, label in [(False, COL_OFF, 'Stim OFF'), (True, COL_ON, 'Stim ON')]:
    grp = all_trials[all_trials['stim_on'] == stim_val]['acc']
    print(f"{label}: n={len(grp)}, acc={grp.mean()*100:.1f}%")

grps = [all_trials[all_trials['stim_on']==v]['acc'].values for v in [False, True]]
means = [g.mean()*100 for g in grps]
sems  = [safe_sem(g)*100 for g in grps]
ns    = [len(g) for g in grps]
labels_bar = [f"Stim OFF\n(n={ns[0]})", f"Stim ON\n(n={ns[1]})"]
bar_with_sem(ax, [0, 1], means, sems, [COL_OFF, COL_ON], labels_bar)

# Stats
if len(grps[0]) >= 2 and len(grps[1]) >= 2:
    t_stat, p_val = stats.ttest_ind(grps[0], grps[1])
    d_val = cohens_d(grps[0], grps[1])
else:
    p_val, d_val = np.nan, np.nan

ax.set_ylim(0, 130)
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.set_title('A. Overall: Stim ON vs OFF', fontsize=11, fontweight='bold')
ax.axhline(50, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax.set_yticks([0, 25, 50, 75, 100])
ax.grid(axis='y', alpha=0.3, zorder=0)
for i, (m, n) in enumerate(zip(means, ns)):
    ax.text(i, m + sems[i] + 3, f"{m:.1f}%", ha='center', fontsize=9, fontweight='bold')
if not np.isnan(p_val):
    add_sig_bracket(ax, 0, 1, max(means)+max(sems)+10, p_val, d_val)

# Panel B: Accuracy by session x stim
ax = axes[1]
sess_list = ['S2', 'S3']
x_pos = np.array([0, 1, 2.5, 3.5])
grp_data = []
for sess in sess_list:
    for stim_val in [False, True]:
        sub = all_trials[(all_trials['session']==sess) & (all_trials['stim_on']==stim_val)]
        grp_data.append(sub['acc'].values)

means_b = [g.mean()*100 if len(g)>0 else 0 for g in grp_data]
sems_b  = [safe_sem(g)*100 for g in grp_data]
ns_b    = [len(g) for g in grp_data]
colors_b = [COL_OFF, COL_ON, COL_OFF, COL_ON]
xlabels_b = [f"S2 OFF\n(n={ns_b[0]})", f"S2 ON\n(n={ns_b[1]})",
             f"S3 OFF\n(n={ns_b[2]})", f"S3 ON\n(n={ns_b[3]})"]
bar_with_sem(ax, x_pos, means_b, sems_b, colors_b, xlabels_b)

for i, (m, s) in enumerate(zip(means_b, sems_b)):
    ax.text(x_pos[i], m + s + 2, f"{m:.0f}%", ha='center', fontsize=8, fontweight='bold')

# Stats within each session
for i, (sess, x1, x2) in enumerate([('S2', 0, 1), ('S3', 2.5, 3.5)]):
    g_off = all_trials[(all_trials['session']==sess)&(~all_trials['stim_on'])]['acc'].values
    g_on  = all_trials[(all_trials['session']==sess)&(all_trials['stim_on'])]['acc'].values
    if len(g_off)>=2 and len(g_on)>=2:
        _, pv = stats.ttest_ind(g_off, g_on)
        dv = cohens_d(g_off, g_on)
        add_sig_bracket(ax, x1, x2, max(means_b[i*2:i*2+2])+max(sems_b[i*2:i*2+2])+8, pv, dv)

ax.set_ylim(0, 140)
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.set_title('B. By Session: Stim ON vs OFF', fontsize=11, fontweight='bold')
ax.axhline(50, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax.set_yticks([0, 25, 50, 75, 100])
ax.grid(axis='y', alpha=0.3, zorder=0)

# Panel C: Accuracy by num_digits (colored by stim)
ax = axes[2]
digit_combos = [(2, False), (2, True), (3, False), (3, True), (4, False), (4, True)]
x_positions  = [0, 1, 2.5, 3.5, 5, 6]
means_c, sems_c, ns_c, colors_c, xlabels_c = [], [], [], [], []
for (nd, stim_v), xp in zip(digit_combos, x_positions):
    sub = all_trials[(all_trials['num_digits']==nd) & (all_trials['stim_on']==stim_v)]
    if len(sub) == 0:
        means_c.append(np.nan); sems_c.append(0); ns_c.append(0)
    else:
        means_c.append(sub['acc'].mean()*100); sems_c.append(safe_sem(sub['acc'].values)*100)
        ns_c.append(len(sub))
    colors_c.append(COL_ON if stim_v else COL_OFF)
    label_str = f"{nd}d {'ON' if stim_v else 'OFF'}\n(n={ns_c[-1]})"
    xlabels_c.append(label_str)

valid = [not np.isnan(m) for m in means_c]
xp_v = [x_positions[i] for i in range(len(valid)) if valid[i]]
means_v = [means_c[i] for i in range(len(valid)) if valid[i]]
sems_v  = [sems_c[i]  for i in range(len(valid)) if valid[i]]
cols_v  = [colors_c[i] for i in range(len(valid)) if valid[i]]
labels_v = [xlabels_c[i] for i in range(len(valid)) if valid[i]]

bar_with_sem(ax, xp_v, means_v, sems_v, cols_v, labels_v)
for xp, m, s in zip(xp_v, means_v, sems_v):
    ax.text(xp, m+s+2, f"{m:.0f}%", ha='center', fontsize=7.5, fontweight='bold')

ax.set_ylim(0, 140)
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.set_title('C. By Difficulty × Stim', fontsize=11, fontweight='bold')
ax.axhline(50, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax.set_yticks([0, 25, 50, 75, 100])
ax.grid(axis='y', alpha=0.3, zorder=0)
# Add digit span labels
for x_grp, nd in [(0.5, '2-Digit'), (3.0, '3-Digit'), (5.5, '4-Digit')]:
    ax.text(x_grp, -18, nd, ha='center', fontsize=9, fontweight='bold', color='#333333')

plt.tight_layout()
plt.savefig('fig1_stim_accuracy_overall.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
plt.close(fig)
print("Figure 1 saved.")


# ══════════════════════════════════════════════
# FIGURE 2: Hard Trials (Session 2 / 4-digit) — Stim ON vs OFF
# ══════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(15, 6), facecolor='white')
fig.suptitle('Session 2 — 4-Digit Trials: Stimulation Effect',
             fontsize=14, fontweight='bold')

s2_4d = trials_s2[trials_s2['num_digits'] == 4].copy()
on_4d  = s2_4d[s2_4d['stim_on']]
off_4d = s2_4d[~s2_4d['stim_on']]

print(f"\nS2 4-digit: Stim ON n={len(on_4d)}, OFF n={len(off_4d)}")
print(f"  Stim ON  acc={on_4d['acc'].mean()*100:.1f}%  RT={on_4d['rt_ms'].mean():.0f}ms")
print(f"  Stim OFF acc={off_4d['acc'].mean()*100:.1f}%  RT={off_4d['rt_ms'].mean():.0f}ms")

# Panel A: Accuracy
ax = axes[0]
grps = [off_4d['acc'].values, on_4d['acc'].values]
means = [g.mean()*100 for g in grps]
sems  = [safe_sem(g)*100 for g in grps]
ns    = [len(g) for g in grps]
xlabs = [f"Stim OFF\n(n={ns[0]})", f"Stim ON\n(n={ns[1]})"]
bar_with_sem(ax, [0,1], means, sems, [COL_OFF, COL_ON], xlabs)
for i,(m,s) in enumerate(zip(means,sems)):
    ax.text(i, m+s+2, f"{m:.1f}%", ha='center', fontsize=10, fontweight='bold')

if len(grps[0])>=2 and len(grps[1])>=2:
    _, pv = stats.ttest_ind(grps[0], grps[1])
    dv = cohens_d(grps[0], grps[1])
    add_sig_bracket(ax, 0, 1, max(means)+max(sems)+8, pv, dv)
    print(f"  Accuracy t-test: p={pv:.3f}, d={dv:.2f}")

ax.set_ylim(0, 130); ax.set_yticks([0,25,50,75,100])
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.set_title('A. 4-Digit Accuracy', fontsize=11, fontweight='bold')
ax.axhline(50, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax.grid(axis='y', alpha=0.3)

# Panel B: RT (correct trials only)
ax = axes[1]
on_4d_c  = on_4d[on_4d['acc']==1]
off_4d_c = off_4d[off_4d['acc']==1]
grps_rt  = [off_4d_c['rt_ms'].values, on_4d_c['rt_ms'].values]
means_rt = [g.mean() for g in grps_rt]
sems_rt  = [safe_sem(g) for g in grps_rt]
ns_rt    = [len(g) for g in grps_rt]
xlabs_rt = [f"Stim OFF\n(n={ns_rt[0]})", f"Stim ON\n(n={ns_rt[1]})"]
bar_with_sem(ax, [0,1], means_rt, sems_rt, [COL_OFF, COL_ON], xlabs_rt)
for i,(m,s) in enumerate(zip(means_rt,sems_rt)):
    ax.text(i, m+s+20, f"{m:.0f}ms", ha='center', fontsize=10, fontweight='bold')

if len(grps_rt[0])>=2 and len(grps_rt[1])>=2:
    _, pv_rt = stats.ttest_ind(grps_rt[0], grps_rt[1])
    dv_rt = cohens_d(grps_rt[0], grps_rt[1])
    add_sig_bracket(ax, 0, 1, max(means_rt)+max(sems_rt)+100, pv_rt, dv_rt)

ax.set_ylabel('Response Time (ms)', fontsize=11)
ax.set_title('B. 4-Digit RT (Correct Trials)', fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Panel C: Trial-by-trial accuracy timeline for S2 4-digit
ax = axes[2]
s2_4d_sorted = s2_4d.sort_values('trial_idx').reset_index(drop=True)
colors_timeline = [COL_ON if v else COL_OFF for v in s2_4d_sorted['stim_on']]
ax.bar(range(len(s2_4d_sorted)), s2_4d_sorted['acc']*100,
       color=colors_timeline, edgecolor='black', linewidth=0.8, width=0.7, zorder=3)
ax.set_xticks(range(len(s2_4d_sorted)))
ax.set_xticklabels([f"B{r['block']}T{r['trial']}" for _, r in s2_4d_sorted.iterrows()],
                    fontsize=8, rotation=45)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_yticks([0, 100])
ax.set_yticklabels(['Incorrect', 'Correct'])
ax.set_title('C. 4-Digit Trial Timeline (Blue=OFF, Red=ON)', fontsize=10, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Annotate mA
for i, (_, row) in enumerate(s2_4d_sorted.iterrows()):
    ax.text(i, 5, f"{row['mean_mA']:.1f}mA", ha='center', fontsize=7.5,
            rotation=90, va='bottom', color='white' if row['acc']==1 else 'black')

plt.tight_layout()
plt.savefig('fig2_s2_4digit_stim.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
plt.close(fig)
print("Figure 2 saved.")


# ══════════════════════════════════════════════
# FIGURE 3: Stimulation Timing Window Analysis
# Question: Does stim concentrated in ENCODING vs RECALL phase affect accuracy?
# ══════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(15, 6), facecolor='white')
fig.suptitle('Stimulation Timing Window — Encoding vs Recall Phase',
             fontsize=14, fontweight='bold')

# Use all trials (both sessions)
# Classify each trial by WHICH phase had higher stim
def classify_stim_phase(row):
    if row['mA_enc'] < 0.5 and row['mA_rec'] < 0.5:
        return 'No Stim'
    if row['mA_enc'] >= STIM_THRESH_MA and row['mA_rec'] >= STIM_THRESH_MA:
        return 'Both'
    if row['mA_enc'] >= STIM_THRESH_MA:
        return 'Encoding Only'
    if row['mA_rec'] >= STIM_THRESH_MA:
        return 'Recall Only'
    # Sub-threshold: compare which phase had more
    if row['mA_enc'] > row['mA_rec']:
        return 'More in Encoding'
    else:
        return 'More in Recall'

all_trials['stim_phase'] = all_trials.apply(classify_stim_phase, axis=1)
print("\nStim phase distribution:")
print(all_trials['stim_phase'].value_counts())

# Panel A: Accuracy by stim phase (threshold-based categories)
ax = axes[0]
# Main categories
phase_cats = ['No Stim', 'Encoding Only', 'Recall Only', 'Both']
phase_colors = ['#AAAAAA', '#4393C3', '#D6604D', '#8B4513']
ph_means, ph_sems, ph_ns = [], [], []
for cat in phase_cats:
    sub = all_trials[all_trials['stim_phase']==cat]
    if len(sub) > 0:
        ph_means.append(sub['acc'].mean()*100)
        ph_sems.append(safe_sem(sub['acc'].values)*100)
        ph_ns.append(len(sub))
    else:
        ph_means.append(np.nan); ph_sems.append(0); ph_ns.append(0)

valid_ph = [(i,c) for i,c in enumerate(phase_cats) if ph_ns[i]>0]
xp_ph = list(range(len(valid_ph)))
for j, (i, cat) in enumerate(valid_ph):
    ax.bar(j, ph_means[i], color=phase_colors[i], edgecolor='black',
           linewidth=1.0, width=0.6, zorder=3)
    ax.errorbar(j, ph_means[i], yerr=ph_sems[i],
                fmt='none', color='black', capsize=5, capthick=1.5, zorder=4)
    ax.text(j, ph_means[i]+ph_sems[i]+2,
            f"{ph_means[i]:.0f}%\n(n={ph_ns[i]})",
            ha='center', fontsize=8.5, fontweight='bold')

ax.set_xticks(range(len(valid_ph)))
ax.set_xticklabels([c for _,c in valid_ph], fontsize=9, rotation=20, ha='right')
ax.set_ylim(0, 130)
ax.set_yticks([0, 25, 50, 75, 100])
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.set_title('A. Accuracy by Stim Phase', fontsize=11, fontweight='bold')
ax.axhline(50, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax.grid(axis='y', alpha=0.3)

# Panel B: Continuous mA in encoding vs recall — scatter colored by accuracy
ax = axes[1]
for acc_v, color, label, marker in [(1, '#2166AC', 'Correct', 'o'), (0, '#D6604D', 'Incorrect', 'X')]:
    sub = all_trials[all_trials['acc']==acc_v]
    ax.scatter(sub['mA_enc'], sub['mA_rec'], c=color, label=label,
               s=80, alpha=0.75, marker=marker, edgecolors='black', linewidths=0.5, zorder=3)

ax.axvline(STIM_THRESH_MA, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax.axhline(STIM_THRESH_MA, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax.text(STIM_THRESH_MA+0.1, 0.1, f'Thresh={STIM_THRESH_MA}mA', fontsize=8, color='gray')
ax.set_xlabel('Mean mA — Encoding Phase', fontsize=10)
ax.set_ylabel('Mean mA — Recall Phase', fontsize=10)
ax.set_title('B. Encoding vs Recall mA\n(by correctness)', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Add quadrant labels
ax_xlim = ax.get_xlim(); ax_ylim = ax.get_ylim()
for qx, qy, label in [
    (0.3, 0.3, 'Low/Low'),
    (STIM_THRESH_MA+0.5, 0.3, 'Enc ON\nRec OFF'),
    (0.3, STIM_THRESH_MA+0.3, 'Enc OFF\nRec ON'),
    (STIM_THRESH_MA+0.5, STIM_THRESH_MA+0.3, 'Both ON')
]:
    ax.text(qx, qy, label, fontsize=7.5, color='#555555', ha='left',
            style='italic', zorder=2)

# Panel C: mA over trial time — binned into 5 windows, correct vs incorrect
ax = axes[2]
# Use S3 (only session with per-trial stim variation across window)
# We'll bin each trial's LFP time series into 5 equal windows
# and compare mA profile for correct vs incorrect

# Build mA profile per trial (S2 and S3)
def get_mA_profile(lfp_df, trial_start, trial_end, n_bins=5):
    dur = trial_end - trial_start
    bins = np.linspace(trial_start, trial_end, n_bins+1)
    profile = []
    for i in range(n_bins):
        m = get_mean_mA(lfp_df, bins[i], bins[i+1])
        profile.append(m)
    return np.array(profile)

n_bins = 5
correct_profiles, incorrect_profiles = [], []

for sess, df_trials, lfp_df in [('S2', trials_s2, lfp2), ('S3', trials_s3, lfp3)]:
    SYNC_OFFSET = SYNC_OFFSET_S2 if sess=='S2' else SYNC_OFFSET_S3
    T0 = T0_S2 if sess=='S2' else T0_S3
    for _, row in df_trials.iterrows():
        prof = get_mA_profile(lfp_df, row['fix_abs'] if 'fix_abs' in row.index else
                              rttime_to_abs(row.get('fix_onset_ms', 0), T0, SYNC_OFFSET),
                              row['choice_end_abs'] if 'choice_end_abs' in row.index else 0,
                              n_bins)
        if row['acc'] == 1:
            correct_profiles.append(prof)
        else:
            incorrect_profiles.append(prof)

# Need fix_abs in trials for this — rebuild with correct column names
# Use already-computed abs times from build_trials
for sess, df_t, lfp_df in [('S2', trials_s2, lfp2), ('S3', trials_s3, lfp3)]:
    pass

# Re-extract from the dataframe
correct_profiles, incorrect_profiles = [], []
for sess, df_t, lfp_df_used in [('S2', trials_s2, lfp2), ('S3', trials_s3, lfp3)]:
    T0_used = T0_S2 if sess=='S2' else T0_S3
    OFFSET_used = SYNC_OFFSET_S2 if sess=='S2' else SYNC_OFFSET_S3
    for _, row in df_t.iterrows():
        t_start = rttime_to_abs(row['fix_onset_ms'] if 'fix_onset_ms' in df_t.columns
                                else row.get('fix_abs', T0_used), T0_used, OFFSET_used)
        t_end   = rttime_to_abs(row['rttime'] if 'rttime' in df_t.columns
                                else row.get('choice_end_abs', T0_used), T0_used, OFFSET_used)
        bins = np.linspace(t_start, t_end, n_bins+1)
        prof = np.array([get_mean_mA(lfp_df_used, bins[i], bins[i+1]) for i in range(n_bins)])
        if row['acc'] == 1:
            correct_profiles.append(prof)
        else:
            incorrect_profiles.append(prof)

correct_profiles   = np.array(correct_profiles)
incorrect_profiles = np.array(incorrect_profiles)

bin_labels = ['0-20%', '20-40%', '40-60%', '60-80%', '80-100%']
x_bins = np.arange(n_bins)

c_mean = correct_profiles.mean(axis=0)
c_sem  = np.array([safe_sem(correct_profiles[:,i]) for i in range(n_bins)])
i_mean = incorrect_profiles.mean(axis=0)
i_sem  = np.array([safe_sem(incorrect_profiles[:,i]) for i in range(n_bins)])

ax.plot(x_bins, c_mean, color='#2166AC', linewidth=2.5, marker='o', label=f'Correct (n={len(correct_profiles)})')
ax.fill_between(x_bins, c_mean-c_sem, c_mean+c_sem, color='#2166AC', alpha=0.2)
ax.plot(x_bins, i_mean, color='#D6604D', linewidth=2.5, marker='s', label=f'Incorrect (n={len(incorrect_profiles)})')
ax.fill_between(x_bins, i_mean-i_sem, i_mean+i_sem, color='#D6604D', alpha=0.2)

ax.axhline(STIM_THRESH_MA, color='gray', linestyle='--', linewidth=1.2, alpha=0.7,
           label=f'Stim ON threshold ({STIM_THRESH_MA}mA)')
ax.set_xticks(x_bins)
ax.set_xticklabels(bin_labels, fontsize=9)
ax.set_xlabel('Trial Time Window (% of trial)', fontsize=10)
ax.set_ylabel('Mean mA (Left ATN)', fontsize=10)
ax.set_title('C. mA Profile: Correct vs Incorrect\n(binned trial time)', fontsize=11, fontweight='bold')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('fig3_stim_timing_window.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
plt.close(fig)
print("Figure 3 saved.")


# ══════════════════════════════════════════════
# FIGURE 4: mA Dose vs Accuracy + RT
# Continuous relationship between stim amplitude and performance
# ══════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(15, 6), facecolor='white')
fig.suptitle('Stimulation Amplitude — Dose-Response Relationship',
             fontsize=14, fontweight='bold')

# Panel A: mA distribution — Correct vs Incorrect
ax = axes[0]
corr_mA   = all_trials[all_trials['acc']==1]['mean_mA'].values
incorr_mA = all_trials[all_trials['acc']==0]['mean_mA'].values

bins_mA = np.linspace(0, all_trials['mean_mA'].max()+0.5, 15)
ax.hist(corr_mA, bins=bins_mA, color='#2166AC', alpha=0.65,
        label=f'Correct (n={len(corr_mA)})', edgecolor='white', linewidth=0.5)
ax.hist(incorr_mA, bins=bins_mA, color='#D6604D', alpha=0.65,
        label=f'Incorrect (n={len(incorr_mA)})', edgecolor='white', linewidth=0.5)

ax.axvline(STIM_THRESH_MA, color='black', linestyle='--', linewidth=1.5,
           label=f'Threshold ({STIM_THRESH_MA}mA)')
ax.set_xlabel('Mean Trial mA (Left ATN)', fontsize=10)
ax.set_ylabel('Number of Trials', fontsize=10)
ax.set_title('A. mA Distribution\nCorrect vs Incorrect', fontsize=11, fontweight='bold')
ax.legend(fontsize=8.5)
ax.grid(alpha=0.3)

t_mA, p_mA = stats.ttest_ind(corr_mA, incorr_mA) if (len(corr_mA)>=2 and len(incorr_mA)>=2) else (np.nan, np.nan)
d_mA = cohens_d(corr_mA, incorr_mA)
ax.text(0.98, 0.95, f"t-test p={p_mA:.3f}\nd={d_mA:.2f}" if not np.isnan(p_mA) else "",
        transform=ax.transAxes, ha='right', va='top', fontsize=8.5,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Panel B: mA binned into quartiles vs accuracy
ax = axes[1]
all_trials['mA_quartile'] = pd.qcut(all_trials['mean_mA'], q=4,
                                     labels=['Q1\n(Low)', 'Q2', 'Q3', 'Q4\n(High)'])
qrt_acc = all_trials.groupby('mA_quartile', observed=True)['acc'].agg(['mean','count','sem'])
qrt_acc['sem'] = qrt_acc.apply(lambda row: row['sem'] if row['count']>1 else 0, axis=1)
mA_qrt_ranges = all_trials.groupby('mA_quartile', observed=True)['mean_mA'].agg(['min','max'])

bars = ax.bar(range(len(qrt_acc)), qrt_acc['mean']*100,
              color=['#4393C3','#92C5DE','#F4A582','#D6604D'],
              edgecolor='black', linewidth=1.0, width=0.6, zorder=3)
ax.errorbar(range(len(qrt_acc)), qrt_acc['mean']*100,
            yerr=qrt_acc['sem']*100, fmt='none', color='black',
            capsize=5, capthick=1.5, zorder=4)
ax.set_xticks(range(len(qrt_acc)))
ax.set_xticklabels([f"{idx}\n({mA_qrt_ranges.loc[idx,'min']:.1f}-{mA_qrt_ranges.loc[idx,'max']:.1f}mA)"
                    for idx in qrt_acc.index], fontsize=8.5)
for i, (acc, n) in enumerate(zip(qrt_acc['mean']*100, qrt_acc['count'])):
    ax.text(i, acc + qrt_acc['sem'].iloc[i]*100 + 2,
            f"{acc:.0f}%\n(n={n})", ha='center', fontsize=8.5, fontweight='bold')

ax.set_ylim(0, 130)
ax.set_yticks([0, 25, 50, 75, 100])
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.set_xlabel('mA Quartile', fontsize=10)
ax.set_title('B. Accuracy by mA Quartile', fontsize=11, fontweight='bold')
ax.axhline(50, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax.grid(axis='y', alpha=0.3)

# Spearman correlation mA vs acc
rho, p_rho = stats.spearmanr(all_trials['mean_mA'], all_trials['acc'])
ax.text(0.98, 0.05, f"Spearman ρ={rho:.3f}\np={p_rho:.3f}",
        transform=ax.transAxes, ha='right', va='bottom', fontsize=8.5,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Panel C: mA vs RT scatter (correct trials only)
ax = axes[2]
corr_all = all_trials[all_trials['acc']==1]
for sess, color, marker in [('S2', COL_S2, 'o'), ('S3', COL_S3, 's')]:
    sub = corr_all[corr_all['session']==sess]
    ax.scatter(sub['mean_mA'], sub['rt_ms'], c=color, s=70, label=sess,
               marker=marker, alpha=0.8, edgecolors='black', linewidths=0.5, zorder=3)

# Fit trendline
if len(corr_all) >= 4:
    try:
        z = np.polyfit(corr_all['mean_mA'], corr_all['rt_ms'], 1)
        p_trend = np.poly1d(z)
        x_line = np.linspace(corr_all['mean_mA'].min(), corr_all['mean_mA'].max(), 100)
        ax.plot(x_line, p_trend(x_line), 'k--', linewidth=1.5, alpha=0.6, label='Trend')
        rho_rt, p_rt = stats.spearmanr(corr_all['mean_mA'], corr_all['rt_ms'])
        ax.text(0.98, 0.95, f"ρ={rho_rt:.3f}\np={p_rt:.3f}",
                transform=ax.transAxes, ha='right', va='top', fontsize=8.5,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    except:
        pass

ax.axvline(STIM_THRESH_MA, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax.set_xlabel('Mean Trial mA (Left ATN)', fontsize=10)
ax.set_ylabel('Response Time (ms)', fontsize=10)
ax.set_title('C. mA vs RT (Correct Trials)', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('fig4_mA_dose_response.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
plt.close(fig)
print("Figure 4 saved.")


# ══════════════════════════════════════════════
# FIGURE 5: Summary Metrics Table
# ══════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(14, 7), facecolor='white')
ax.axis('off')

# Build summary table
rows = []

# 1. Overall stim ON vs OFF
for stim_v, label in [(False,'Stim OFF (All)'),(True,'Stim ON (All)')]:
    sub = all_trials[all_trials['stim_on']==stim_v]
    if len(sub) > 0:
        rows.append([label, f"Both", 'All', f"{sub['mean_mA'].mean():.2f}",
                     f"{len(sub)}", f"{sub['acc'].mean()*100:.1f}%",
                     f"{sub[sub['acc']==1]['rt_ms'].mean():.0f}ms" if len(sub[sub['acc']==1])>0 else "N/A"])

# 2. Per session per stim
for sess in ['S2','S3']:
    for stim_v, stim_l in [(False,'OFF'),(True,'ON')]:
        sub = all_trials[(all_trials['session']==sess)&(all_trials['stim_on']==stim_v)]
        if len(sub) > 0:
            rt_c = sub[sub['acc']==1]['rt_ms']
            rows.append([f"Stim {stim_l}", sess,
                         str(sorted(sub['num_digits'].unique())),
                         f"{sub['mean_mA'].mean():.2f}",
                         f"{len(sub)}",
                         f"{sub['acc'].mean()*100:.1f}%",
                         f"{rt_c.mean():.0f}ms" if len(rt_c)>0 else "N/A"])

# 3. S2 4-digit breakdown
for stim_v, stim_l in [(False,'OFF'),(True,'ON')]:
    sub = all_trials[(all_trials['session']=='S2')&(all_trials['num_digits']==4)&(all_trials['stim_on']==stim_v)]
    if len(sub)>0:
        rt_c = sub[sub['acc']==1]['rt_ms']
        rows.append([f"S2 4-digit Stim {stim_l}", 'S2', '[4]',
                     f"{sub['mean_mA'].mean():.2f}", f"{len(sub)}",
                     f"{sub['acc'].mean()*100:.1f}%",
                     f"{rt_c.mean():.0f}ms" if len(rt_c)>0 else "N/A"])

# Stats rows
rows.append(['─'*20,'─'*5,'─'*7,'─'*6,'─'*4,'─'*10,'─'*8])
g_off = all_trials[~all_trials['stim_on']]['acc'].values
g_on  = all_trials[all_trials['stim_on']]['acc'].values
if len(g_off)>=2 and len(g_on)>=2:
    _, pv = stats.ttest_ind(g_off, g_on)
    dv = cohens_d(g_off, g_on)
    rows.append([f"Stim ON vs OFF (all)", 'Stats', '—', '—', '—',
                 f"t-test p={pv:.3f}", f"Cohen's d={dv:.2f}"])

rho_v, p_rho_v = stats.spearmanr(all_trials['mean_mA'], all_trials['acc'])
rows.append([f"mA vs Accuracy", 'Stats', '—', '—', '—',
             f"Spearman ρ={rho_v:.3f}", f"p={p_rho_v:.3f}"])

headers = ['Condition', 'Session', 'Digits', 'Mean mA', 'N', 'Accuracy', 'RT (corr.)']
table = ax.table(cellText=rows, colLabels=headers,
                 loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.6)

# Color header
for j in range(len(headers)):
    table[0, j].set_facecolor('#2C3E50')
    table[0, j].set_text_props(color='white', fontweight='bold')

# Highlight stim ON rows
stim_on_rows = [i+1 for i,r in enumerate(rows) if 'Stim ON' in str(r[0])]
for ri in stim_on_rows:
    for j in range(len(headers)):
        table[ri, j].set_facecolor('#FDECEA')

ax.set_title('Summary Metrics — Stimulation × Accuracy Analysis',
             fontsize=13, fontweight='bold', pad=10)

plt.tight_layout()
plt.savefig('fig5_summary_table.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
plt.close(fig)
print("Figure 5 (summary table) saved.")

print("\n" + "="*60)
print("ALL FIGURES SAVED.")
print("="*60)
print("\nKEY FINDINGS:")
print(f"  • Stim ON trials (>=2mA): {all_trials['stim_on'].sum()}/{len(all_trials)} total")
print(f"  • Stim ON accuracy: {all_trials[all_trials['stim_on']]['acc'].mean()*100:.1f}%")
print(f"  • Stim OFF accuracy: {all_trials[~all_trials['stim_on']]['acc'].mean()*100:.1f}%")
g1 = all_trials[all_trials['stim_on']]['acc'].values
g2 = all_trials[~all_trials['stim_on']]['acc'].values
if len(g1)>=2 and len(g2)>=2:
    _, pv = stats.ttest_ind(g1, g2)
    print(f"  • t-test p={pv:.3f}, Cohen's d={cohens_d(g1,g2):.2f}")
rho_f, p_rho_f = stats.spearmanr(all_trials['mean_mA'], all_trials['acc'])
print(f"  • Spearman mA-accuracy: ρ={rho_f:.3f}, p={p_rho_f:.3f}")

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\ASSUS\\ATN\\Digit Span Backwards\\Data\\Eprime Data\\Digit Span Backwards v3.2\\Events.csv'